## Results viewer (`pro.results()`)

Prototype of the interactive viewer of simulation results.

- **Left panel**: components grouped by type. Click a type and then a component to see its variables, and click a variable to plot it (up to 8 at the same time). The filter box searches by "component.variable".
- **Toolbar**: frequency (time step, hourly, daily, monthly, yearly), aggregation used when resampling (mean, sum, maximum, minimum) and interval (From / To).
- **Charts**: one per unit, with the cursor synchronised between them. Drag horizontally on a chart to zoom, double-click to go back.
- **Height** changes the height of the charts. Hovering near a line highlights it and dims the others; clicking a series in the legend hides or shows it.
- **Table** shows the values as a table, **CSV** downloads them, **Clear** removes the selection.

In [ ]:
import os
import opensimula as osm

# The ASHRAE 140 case files refer to the weather files relative to the test folder
os.chdir("../ASHRAE_140/LOAD_TEST")

### Case 600: one zone with south windows

The viewer can start with some variables already selected, as "component.variable".

In [ ]:
sim = osm.Simulation()
pro = sim.new_project("Case 600")
pro.read_json("cases/case_600.json")
pro.simulate()

viewer = pro.results("spaces_1.temperature", "Denver.temperature", "system.Q_sensible")
viewer

### Controlling it from Python

The same options can be set from code; the widget above updates.

In [ ]:
viewer.frequency = "monthly"
viewer.aggregation = "mean"

In [ ]:
viewer.interval = ["2001-07-01", "2001-07-07"]
viewer.frequency = "none"

`dataframe()` returns what the viewer is showing (resampled and cut to the interval) as a pandas DataFrame.

In [ ]:
viewer.dataframe().head()

The heights can also be set from Python, when creating the viewer (`pro.results(..., height=700, chart_height=400)`) or afterwards:

In [ ]:
viewer.chart_height = 400
viewer.height = 750

### Case 960: two zones (sunspace)

A new viewer of another project. Try selecting the temperatures of `back_zone` and `sun_zone` and the loads of `system`.

In [ ]:
pro960 = sim.new_project("Case 960")
pro960.read_json("cases/case_960.json")
pro960.simulate()

pro960.results()

### 3D view

The **3D** button of the toolbar (or `show_3d=True`) shows the geometry above the charts:

- Clicking a surface or window opens it, and its space, in the list on the left, ready to pick its variables.
- The surfaces whose variables are plotted are drawn in the colour of their series; the outline of a space plotted takes its colour too.
- Drag to rotate, wheel to zoom the model; the camera is kept when the colours change.

Plotly is sent from Python the first time a 3D view is opened in the page (about 1.7 MB), so it also works in VS Code.

In [ ]:
pro960.results("back_zone.temperature", "sun_zone.temperature", "interior_wall.T_s0", show_3d=True)

#### Colouring the model by a variable

The **Colour** selector of the 3D view, or from Python. The value of each surface is the mean, maximum or minimum over the interval shown; heat flows with both signs get a diverging scale (blue negative, red positive).

In [ ]:
viewer960 = pro960.results("sun_zone.temperature", show_3d=True)
viewer960.interval = ["2001-01-01", "2001-01-31"]
viewer960.color_variable = "q_sol1"   # solar absorbed on the inside face
viewer960.color_statistic = "max"
viewer960

#### Following the cursor of the charts

With **At cursor**, moving the mouse over the charts colours the model with the values at that time (shown next to the selector). The scale is fixed for the whole interval.

In [ ]:
viewer_t = pro960.results("sun_zone.temperature", "back_zone.temperature", show_3d=True)
viewer_t.interval = ["2001-01-10", "2001-01-12"]
viewer_t.color_variable = "T_s1"
viewer_t.color_statistic = "instant"
viewer_t

### Marimo

In marimo the viewer is wrapped like the project editor:

```python
viewer = mo.ui.anywidget(pro.results("spaces_1.temperature"))
```